# 01 Core Strategy Factory (фабрика базовых стратегий)

Этот notebook (ноутбук) специально сделан легким для браузера: тяжелый код лежит в `core_strategy_factory.py (файл Python)`, а здесь остается только короткий сценарий запуска.

Рабочие files (файлы) notebook (ноутбука) лежат на диске C, потому что Jupyter (Юпитер) запускается оттуда.

Все results (результаты) сохраняются на диск D:

`D:\Filter_2_option_v2\01_core_strategy_factory_output`

Исходный большой файл не копируется:

`./data\dfs_5min.pickle`

## Какие стратегии генерируются

Фабрика создает разные `core strategies (базовые стратегии)`, чтобы следующий `two-option filter (двухопционный фильтр)` не был просто усилителем заранее красивых стратегий:

- `time series momentum (моментум временного ряда)`;
- `time series reversal (реверс временного ряда)`;
- `moving average crossover (пересечение скользящих средних)`;
- `channel breakout (пробой канала)`;
- `channel reversal (реверс от границ канала)`;
- `pair spread mean reversion (возврат парного спреда к среднему)`;
- `pair spread momentum (моментум парного спреда)`;
- `sector trend portfolio (секторный трендовый портфель)`;
- `sector cross sectional momentum (секторный поперечный моментум)`;
- `sector cross sectional reversal (секторный поперечный реверс)`.

`calendar spreads (календарные спреды)` пока не строятся, потому что в исходных данных есть только один активный контракт в каждый момент времени.

In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path(r"./workspace")
sys.path.insert(0, str(WORK_ROOT))

import core_strategy_factory as factory

print("Output (папка результатов):", factory.OUTPUT_DIR)
print("Numba (JIT-компилятор Python):", factory.HAS_NUMBA)
factory.describe_strategy_families()

## Запуск фабрики

Теперь фабрика работает в `batch mode (пакетном режиме)`. Один запуск обрабатывает только выбранные `100 strategies (100 стратегий)`, сохраняет их на диск D и завершает отдельный Python process (процесс Python). Это важно: память полностью освобождается после каждого пакета.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(WORK_ROOT / "core_strategy_factory.py"),
    "--mode", "summary",
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [ ]:
BATCH_ID = 0
BATCH_SIZE = 100

cmd = [
    sys.executable,
    str(WORK_ROOT / "core_strategy_factory.py"),
    "--mode", "batch",
    "--batch-id", str(BATCH_ID),
    "--batch-size", str(BATCH_SIZE),
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Если случайно менялся `BATCH_SIZE (размер пакета)`, лучше догонять пропуски через явный диапазон `START_ID/END_ID (начальный/конечный идентификатор)`, а не через `BATCH_ID (номер пакета)`.

In [ ]:
START_ID = 300
END_ID = 450
BATCH_LABEL = f"repair_{START_ID}_{END_ID}"

cmd = [
    sys.executable,
    str(WORK_ROOT / "core_strategy_factory.py"),
    "--mode", "batch",
    "--start-id", str(START_ID),
    "--end-id", str(END_ID),
    "--batch-label", BATCH_LABEL,
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [ ]:
cmd = [
    sys.executable,
    str(WORK_ROOT / "core_strategy_factory.py"),
    "--mode", "audit",
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [ ]:
cmd = [
    sys.executable,
    str(WORK_ROOT / "core_strategy_factory.py"),
    "--mode", "combine",
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

## Главные файлы после запуска

- `02_strategy_spec_catalog.csv` - полный каталог потенциальных стратегий.
- `batches/metrics_batch_XXXX.csv` - метрики одного `batch (пакета)`.
- `batches/paths_batch_XXXX.csv` - пути к `PnL paths (траекториям прибыли и убытка)` одного `batch (пакета)`.
- `03_core_strategy_train_metrics.csv` - объединенные стратегии и метрики после `combine (объединения)`.
- `04_core_strategy_catalog.csv` - компактный каталог.
- `06_all_core_strategy_paths.csv` - пути ко всем сохраненным `PnL paths (траекториям прибыли и убытка)`.
- `07_manifest.json` - паспорт запуска.